# OCR Fundamentals with Tesseract

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null!pip install -q pytesseract opencv-python-headless pillow

In [ ]:
import cv2import numpy as npimport pytesseractfrom pytesseract import Outputfrom PIL import Image, ImageDraw, ImageFont

In [ ]:
DATASET_PATH = ""TESSERACT_CONFIG = "--oem 3 --psm 6"

In [ ]:
def create_sample_image(text, width=800, height=200, font_size=28):    image = Image.new("RGB", (width, height), color=(255, 255, 255))    draw = ImageDraw.Draw(image)    try:        font = ImageFont.truetype("DejaVuSans.ttf", font_size)    except IOError:        font = ImageFont.load_default()    draw.text((20, height // 2 - font_size), text, fill=(0, 0, 0), font=font)    return imagesample_text = "Tesseract OCR converts images of text into machine readable text."image = create_sample_image(sample_text)image

In [ ]:
def to_grayscale(image):    return cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)def denoise(gray):    return cv2.medianBlur(gray, 3)def binarize(gray):    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)    return threshdef deskew(binary):    coords = np.column_stack(np.where(binary > 0))    angle = cv2.minAreaRect(coords)[-1]    if angle < -45:        angle = -(90 + angle)    else:        angle = -angle    (h, w) = binary.shape    center = (w // 2, h // 2)    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)    rotated = cv2.warpAffine(binary, matrix, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)    return rotated

In [ ]:
def preprocess_image(image):    gray = to_grayscale(image)    denoised = denoise(gray)    binary = binarize(denoised)    return binarypreprocessed = preprocess_image(image)Image.fromarray(preprocessed)

In [ ]:
raw_text = pytesseract.image_to_string(image, config=TESSERACT_CONFIG)preprocessed_text = pytesseract.image_to_string(preprocessed, config=TESSERACT_CONFIG)print("raw:", raw_text.strip())print("preprocessed:", preprocessed_text.strip())

In [ ]:
data = pytesseract.image_to_data(preprocessed, config=TESSERACT_CONFIG, output_type=Output.DICT)for i in range(len(data["text"])):    if data["text"][i].strip():        print(data["text"][i], data["conf"][i], (data["left"][i], data["top"][i], data["width"][i], data["height"][i]))

In [ ]:
def draw_boxes(image, data, conf_threshold=0):    boxed = image.convert("RGB").copy()    draw = ImageDraw.Draw(boxed)    for i in range(len(data["text"])):        if data["text"][i].strip() and int(data["conf"][i]) >= conf_threshold:            x, y, w, h = data["left"][i], data["top"][i], data["width"][i], data["height"][i]            draw.rectangle([x, y, x + w, y + h], outline=(255, 0, 0), width=2)    return boxedboxed_image = draw_boxes(Image.fromarray(preprocessed), data)boxed_image

In [ ]:
def ocr_pipeline(image_path=None, pil_image=None, lang="eng"):    if pil_image is None:        pil_image = Image.open(image_path)    preprocessed = preprocess_image(pil_image)    text = pytesseract.image_to_string(preprocessed, lang=lang, config=TESSERACT_CONFIG)    data = pytesseract.image_to_data(preprocessed, lang=lang, config=TESSERACT_CONFIG, output_type=Output.DICT)    confidences = [int(c) for c in data["conf"] if c != "-1"]    avg_conf = sum(confidences) / len(confidences) if confidences else 0.0    return {"text": text.strip(), "avg_confidence": avg_conf, "data": data}

In [ ]:
if DATASET_PATH:    result = ocr_pipeline(image_path=DATASET_PATH)else:    result = ocr_pipeline(pil_image=image)print(result["text"])print(result["avg_confidence"])